# Gold Aggregates — Purpose-Built Analytical Tables

This notebook produces the four gold tables that directly feed the project's dashboards and headline analyses. Bronze landed raw data, silver shaped it into analytically usable form, and now gold answers specific business questions.

**The difference between silver and gold:**

Silver tables are *general-purpose* — `silver.providers` doesn't know what question I'm asking, it just gives me one row per NPI with everything cleaned up. Gold tables are *purpose-built* — `gold.provider_spending_summary` exists specifically to answer "who are the highest-spending Medicare providers and how do they rank against each other?" That intent shapes the columns I include, the derived metrics I pre-compute, and how I denormalize.

**The four gold tables and what each answers:**

| Gold table | Grain | The business question it answers |
|---|---|---|
| `gold.provider_spending_summary` | one row per NPI | Who are the highest-spending providers? How do they rank by cost, volume, and intensity? |
| `gold.specialty_benchmarks` | one row per specialty | Which specialties cost the most per provider? What's the spending distribution within each specialty? |
| `gold.state_spending_quality` | one row per state | Which states have the highest Medicare spending? How does state-level quality correlate? |
| `gold.hospital_value_scorecard` | one row per hospital (CCN) | What's the relationship between physician spending at a hospital and its measured quality? |


**Why these four:**

The Phase 1 project plan defined these four tables, and after building silver I still think they're the right cuts. Each one answers a real cost-versus-quality question — who pays the most, where the variation lives, what the relationship between spending and outcomes actually looks like. Each one denormalizes the silver layer in a way that's specific to that question — the same data, sliced for a specific use.

**Note on the state_spending_quality table:**

State-level analysis works best when comparing real totals rather than bridge-matched subsets. The provider spending side and the hospital quality side don't share a join key at the row level — they're both rolled up to state independently. This is the standard framing CMS itself uses when publishing state-level cost-vs-quality reports. The hospital-level analysis (`hospital_value_scorecard`) is where the bridge matters, not here.

In [0]:
# Setup: imports and constants

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, DateType, TimestampType
)
from pyspark.sql.window import Window

# Catalog and schema constants
CATALOG = "medicare_provider_quality"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# Default schema for writes is gold
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print(f"Catalog:        {CATALOG}")
print(f"Default schema: {GOLD_SCHEMA}")
print(f"Silver schema:  {SILVER_SCHEMA}  (read-only source)")

Catalog:        medicare_provider_quality
Default schema: gold
Silver schema:  silver  (read-only source)


## 1. gold.provider_spending_summary — one row per NPI, dashboard-ready

The provider-level analytical table. Same grain as `silver.providers` but with derived columns that turn it from "the cleaned data" into "the data ready to be plotted on a dashboard."

### What's different from silver.providers

`silver.providers` gives me total payments, services, beneficiaries, and the primary location/specialty for each NPI. To answer "who are the highest-spending providers?" I have to keep computing the same ranks and percentiles in every query. Gold pre-computes them once.

### The derived columns I'm adding

- **`payment_rank_nationwide`** — dense rank of provider by total Medicare payment, descending. The #1 highest-paid provider gets rank 1.
- **`payment_percentile_nationwide`** — percentile rank as a 0-100 number. Useful for "this provider is in the top 1% nationally."
- **`payment_rank_within_specialty`** — same rank but partitioned by specialty. Lets me say "this cardiologist is the #5 highest-paid cardiologist nationally."
- **`payment_rank_within_state`** — same, partitioned by state. "Highest-paid provider in Virginia."
- **`avg_payment_per_service`** — total payment divided by total services. Useful for spotting outliers — a $20K-per-service provider stands out differently from a $50-per-service one.
- **`services_per_beneficiary`** — utilization intensity. High service-per-bene with low unique beneficiary count is a different pattern from low-per-bene high-volume.

### What I'm filtering out

Organizations only — entity_code = 'O' — and providers with zero total payment (data quality artifacts). The provider-level analysis is fundamentally about *individual physicians*, not hospital organizations or labs that happen to have NPIs. Roughly 6% of NPIs in silver.providers are organizations; filtering them out at gold keeps the analytical table focused.

### What this table does NOT try to do

No cost-vs-quality join here. That comes in `gold.hospital_value_scorecard`. This table is purely about provider spending — who, how much, ranked against peers.

In [0]:
# Build gold.provider_spending_summary — individual providers with analytical ranks

silver_providers = spark.table(f"{SILVER_SCHEMA}.providers")

# Filter to individual providers only, with non-zero payment
base = (
    silver_providers
    .filter(F.col("entity_code") == "I")
    .filter(F.col("total_medicare_payment").isNotNull())
    .filter(F.col("total_medicare_payment") > 0)
)

# Window specs for the various ranks
w_nationwide = Window.orderBy(F.col("total_medicare_payment").desc())
w_specialty  = Window.partitionBy("primary_specialty").orderBy(F.col("total_medicare_payment").desc())
w_state      = Window.partitionBy("primary_state").orderBy(F.col("total_medicare_payment").desc())

# Percentile uses percent_rank which gives a 0.0-1.0 value; multiply for readability
w_pct = Window.orderBy(F.col("total_medicare_payment"))

gold_provider_spending = (
    base
    .withColumn("payment_rank_nationwide",
                F.dense_rank().over(w_nationwide))
    .withColumn("payment_percentile_nationwide",
                F.round(F.percent_rank().over(w_pct) * 100, 2))
    .withColumn("payment_rank_within_specialty",
                F.dense_rank().over(w_specialty))
    .withColumn("payment_rank_within_state",
                F.dense_rank().over(w_state))
    .withColumn("avg_payment_per_service",
                F.when(F.col("total_services") > 0,
                       F.col("total_medicare_payment") / F.col("total_services"))
                 .otherwise(None))
    .withColumn("services_per_beneficiary",
                F.when(F.col("total_beneficiaries_unsuppressed") > 0,
                       F.col("total_services") / F.col("total_beneficiaries_unsuppressed"))
                 .otherwise(None))
    .select(
        "npi",
        "last_or_org_name",
        "first_name",
        "credentials",
        "primary_specialty",
        "primary_state",
        "primary_city",
        "total_medicare_payment",
        "total_medicare_allowed",
        "total_submitted_charges",
        "total_services",
        "total_beneficiaries_unsuppressed",
        "unique_hcpcs_codes",
        "avg_payment_per_service",
        "services_per_beneficiary",
        "payment_rank_nationwide",
        "payment_percentile_nationwide",
        "payment_rank_within_specialty",
        "payment_rank_within_state",
    )
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    gold_provider_spending.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("provider_spending_summary")
)

count = spark.table("provider_spending_summary").count()
print(f"Wrote {count:,} rows to {CATALOG}.{GOLD_SCHEMA}.provider_spending_summary")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Wrote 1,113,408 rows to medicare_provider_quality.gold.provider_spending_summary


In [0]:
spark.sql("""
    SELECT 
        payment_rank_nationwide,
        last_or_org_name,
        first_name,
        primary_specialty,
        primary_state,
        total_medicare_payment,
        total_services,
        avg_payment_per_service,
        payment_percentile_nationwide
    FROM provider_spending_summary
    ORDER BY payment_rank_nationwide
    LIMIT 10
""").show(truncate=False)

+-----------------------+----------------+----------+------------------+-------------+----------------------+--------------+-----------------------+-----------------------------+
|payment_rank_nationwide|last_or_org_name|first_name|primary_specialty |primary_state|total_medicare_payment|total_services|avg_payment_per_service|payment_percentile_nationwide|
+-----------------------+----------------+----------+------------------+-------------+----------------------+--------------+-----------------------+-----------------------------+
|1                      |Denny           |Ira       |Nurse Practitioner|AZ           |1.352587838403116E8   |154796.0      |873.7873319744152      |100.0                        |
|2                      |Kinds           |Jorge     |Nurse Practitioner|AZ           |1.2387326191974877E8  |133990.0      |924.4963200220074      |100.0                        |
|3                      |Goss            |Keith     |Podiatry          |AZ           |9.126264835024484E7

## 2. gold.specialty_benchmarks — one row per specialty, distribution-based metrics

The specialty-level benchmark table. Answers: "Which specialties cost the most per provider? What's the spending distribution within each specialty?" One row per `primary_specialty` value (so ~104 rows).

### Why I'm building this from medians and percentiles, not means

Verification of `gold.provider_spending_summary` surfaced a real CMS data pattern: in some specialties — Nurse Practitioner most visibly — a handful of NPIs reflect organizational billing aggregation rather than individual physician earnings. The #1 NP nationally billed $135M, the #2 NP billed $124M, the #3 NP $91M.

If I compute "average payment per Nurse Practitioner" as a simple mean, those few NPIs drag the average up dramatically. The resulting number — say, $50K-$60K — would be technically accurate as an arithmetic mean but completely misleading as a description of what a typical Nurse Practitioner actually bills.

I'm using percentile-based metrics instead. For each specialty:

- **`median_payment_per_provider`** (P50) — the middle provider's total payment
- **`p25_payment_per_provider`** — the 25th percentile
- **`p75_payment_per_provider`** — the 75th percentile  
- **`p95_payment_per_provider`** — the 95th percentile (top tier, but not the extreme tail)
- **`max_payment_per_provider`** — the absolute maximum (still useful for the "what are the outliers" question)

The median is robust against the kind of outliers we surfaced. The P95 captures the high-performing tier without being warped by the extreme tail. The max is preserved for outlier investigation but isn't presented as the "typical" or "high-end" number.

### What other metrics this table includes

- **`provider_count`** — number of individual providers in the specialty (just a count, no outlier risk)
- **`total_specialty_spending`** — sum of all payments. This IS distorted by outliers, but it's still meaningful at the specialty level — the money was real and Medicare paid it. The interpretation is "total dollars flowing through this specialty" rather than "what a provider in this specialty earns."
- **`median_services_per_provider`** and **`median_beneficiaries_per_provider`** — volume-side metrics, also median-based
- **`spending_concentration_ratio`** — top 1% of providers' share of total specialty spending. This is the metric that exposes the organizational-billing pattern most directly: in normal specialties, the top 1% of providers represent maybe 5-15% of total specialty spending; in specialties with the aggregation pattern, it's much higher.

### What this table doesn't try to do

No quality data. Provider-level quality measures don't exist in this project's data (HCAHPS is hospital-level, not provider-level). Specialty-level quality analysis would need a different data source entirely.

In [0]:
# Build gold.specialty_benchmarks — distribution-based metrics per specialty

provider_summary = spark.table("provider_spending_summary")

# First, compute specialty-level total spending so we can derive the concentration ratio
specialty_totals = (
    provider_summary
    .groupBy("primary_specialty")
    .agg(
        F.sum("total_medicare_payment").alias("total_specialty_spending"),
        F.count("*").alias("provider_count"),
    )
)

# Compute the top-1% concentration: how much of total specialty spending comes
# from the top 1% of providers within that specialty.
# Using percentile_approx to find the P99 threshold, then summing above it.
top_1pct_window = Window.partitionBy("primary_specialty").orderBy(F.col("total_medicare_payment").desc())

top_1pct_spending = (
    provider_summary
    .withColumn("within_specialty_rank", F.row_number().over(top_1pct_window))
    .join(specialty_totals.select("primary_specialty", "provider_count"), on="primary_specialty")
    .withColumn("top_1pct_threshold", F.ceil(F.col("provider_count") * 0.01))
    .filter(F.col("within_specialty_rank") <= F.col("top_1pct_threshold"))
    .groupBy("primary_specialty")
    .agg(F.sum("total_medicare_payment").alias("top_1pct_total"))
)

# Percentile aggregations per specialty
percentile_aggs = (
    provider_summary
    .groupBy("primary_specialty")
    .agg(
        F.expr("percentile_approx(total_medicare_payment, 0.25)").alias("p25_payment_per_provider"),
        F.expr("percentile_approx(total_medicare_payment, 0.50)").alias("median_payment_per_provider"),
        F.expr("percentile_approx(total_medicare_payment, 0.75)").alias("p75_payment_per_provider"),
        F.expr("percentile_approx(total_medicare_payment, 0.95)").alias("p95_payment_per_provider"),
        F.max("total_medicare_payment").alias("max_payment_per_provider"),
        F.expr("percentile_approx(total_services, 0.50)").alias("median_services_per_provider"),
        F.expr("percentile_approx(total_beneficiaries_unsuppressed, 0.50)").alias("median_beneficiaries_per_provider"),
    )
)

# Join everything together
gold_specialty_benchmarks = (
    specialty_totals
    .join(percentile_aggs, on="primary_specialty", how="left")
    .join(top_1pct_spending, on="primary_specialty", how="left")
    .withColumn(
        "spending_concentration_top1pct_share",
        F.round(F.col("top_1pct_total") / F.col("total_specialty_spending") * 100, 2)
    )
    .withColumnRenamed("primary_specialty", "specialty")
    .select(
        "specialty",
        "provider_count",
        "total_specialty_spending",
        "median_payment_per_provider",
        "p25_payment_per_provider",
        "p75_payment_per_provider",
        "p95_payment_per_provider",
        "max_payment_per_provider",
        "median_services_per_provider",
        "median_beneficiaries_per_provider",
        "spending_concentration_top1pct_share",
    )
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    gold_specialty_benchmarks.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("specialty_benchmarks")
)

count = spark.table("specialty_benchmarks").count()
print(f"Wrote {count:,} rows to {CATALOG}.{GOLD_SCHEMA}.specialty_benchmarks")

Wrote 94 rows to medicare_provider_quality.gold.specialty_benchmarks


In [0]:
# Compare specialty benchmarks for Nurse Practitioner vs a few comparison specialties
# Look at: median vs max vs concentration ratio
spark.sql("""
    SELECT 
        specialty,
        provider_count,
        median_payment_per_provider,
        p95_payment_per_provider,
        max_payment_per_provider,
        spending_concentration_top1pct_share
    FROM specialty_benchmarks
    WHERE specialty IN (
        'Nurse Practitioner',
        'Internal Medicine',
        'Family Practice',
        'Cardiology',
        'Diagnostic Radiology',
        'Anesthesiology'
    )
    ORDER BY median_payment_per_provider DESC
""").show(truncate=False)

+--------------------+--------------+---------------------------+------------------------+------------------------+------------------------------------+
|specialty           |provider_count|median_payment_per_provider|p95_payment_per_provider|max_payment_per_provider|spending_concentration_top1pct_share|
+--------------------+--------------+---------------------------+------------------------+------------------------+------------------------------------+
|Cardiology          |19399         |108138.18999930561         |584454.0000041752       |6058904.619998168       |10.58                               |
|Diagnostic Radiology|31554         |76277.21999999389          |330940.77999859815      |1.1954936530223409E7    |11.45                               |
|Internal Medicine   |88702         |44993.500000406            |242324.47999814793      |4.317023469976944E7     |13.77                               |
|Family Practice     |78514         |30159.320000061            |174313.179999751 

## 3. gold.state_spending_quality — one row per state, spending + quality side by side

State-level analytical table. Per state: total Medicare spending, provider/service volume, and aggregated quality measures. Answers "Which states have the highest spending? How does that correlate with quality at the state level?"

### The aggregation choice for this table

State grain is where the bridge limitation stops mattering. Provider spending and hospital quality don't need a row-level join here — they get rolled up to state independently and presented side-by-side. This matches how CMS, the Commonwealth Fund, and Kaiser Family Foundation all publish state-level Medicare cost-vs-quality analyses.

The alternative would be to use only bridge-matched providers (9.3% of NPIs) and bridge-matched hospitals (62.6% of CCNs) to keep them linked at the row level. That would produce a much smaller, biased view of each state — missing most of the actual Medicare spending in the state. State-level rollups are more honest with the full data.

### What I'm computing

**Spending side (from `gold.provider_spending_summary`):**
- Total Medicare payment in the state
- Active provider count
- Median payment per provider (using the same median-based approach from `specialty_benchmarks`)
- Spending per beneficiary (computed by summing payments and dividing by summed unsuppressed beneficiaries)

**Quality side (from `silver.hospital_quality` joined to `silver.hospitals` for state):**
- Hospital count in the state
- For each of the top mortality measures, the state-level mean score (weighted average wouldn't help much since denominators are uneven and the table needs to be readable)
- Average hospital overall star rating across hospitals in the state
- Patient experience averages for the two HCAHPS measures

### Why mean (not median) on the quality side

On the spending side, mean is distorted by the organizational-billing outliers, so median is the right choice. On the quality side, the underlying data is already a rate per hospital — there are no $135M-equivalent outlier hospitals. A mortality rate of 13% at a hospital is just 13%, not skewed by aggregation. Mean is appropriate at this grain.

### What this table doesn't try to do

No statistical correlation analysis built in. Just spending and quality side-by-side per state. Correlation analysis (scatter plots, Spearman coefficients) happens in the analysis notebooks, not the gold table — the gold table is the analytical input, not the output.

In [0]:
# Build gold.state_spending_quality — state-level rollup of spending and quality

# --- Spending side: aggregate from gold.provider_spending_summary ---
spending_by_state = (
    spark.table("provider_spending_summary")
    .groupBy("primary_state")
    .agg(
        F.count("*").alias("active_provider_count"),
        F.sum("total_medicare_payment").alias("total_state_payment"),
        F.expr("percentile_approx(total_medicare_payment, 0.50)").alias("median_provider_payment"),
        F.sum("total_services").alias("total_state_services"),
        F.sum("total_beneficiaries_unsuppressed").alias("total_state_beneficiaries_unsuppressed"),
    )
    .withColumn(
        "payment_per_beneficiary",
        F.when(F.col("total_state_beneficiaries_unsuppressed") > 0,
               F.col("total_state_payment") / F.col("total_state_beneficiaries_unsuppressed"))
         .otherwise(None)
    )
    .withColumnRenamed("primary_state", "state")
)

# --- Quality side: aggregate from silver.hospital_quality joined to silver.hospitals ---
# Need silver.hospitals to get the state for each CCN

quality_with_state = (
    spark.table(f"{SILVER_SCHEMA}.hospital_quality").alias("q")
    .join(
        spark.table(f"{SILVER_SCHEMA}.hospitals").alias("h"),
        on=F.col("q.ccn") == F.col("h.ccn"),
        how="inner",
    )
    .select(
        F.col("h.state"),
        F.col("h.ccn"),
        F.col("h.hospital_overall_rating"),
        F.col("q.measure_id"),
        F.col("q.score"),
    )
)

# Hospital count and average star rating per state
hospital_summary_by_state = (
    spark.table(f"{SILVER_SCHEMA}.hospitals")
    .groupBy("state")
    .agg(
        F.count("*").alias("hospital_count"),
        F.avg("hospital_overall_rating").alias("avg_hospital_overall_rating"),
    )
)

# Pivot the measures into one column per measure
# Using grouping + when() rather than .pivot() for column-name control
quality_by_state = (
    quality_with_state
    .groupBy("state")
    .agg(
        F.avg(F.when(F.col("measure_id") == "MORT_30_AMI", F.col("score"))).alias("avg_mort_30_ami"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_HF", F.col("score"))).alias("avg_mort_30_hf"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_PN", F.col("score"))).alias("avg_mort_30_pn"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_COPD", F.col("score"))).alias("avg_mort_30_copd"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_STK", F.col("score"))).alias("avg_mort_30_stk"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_CABG", F.col("score"))).alias("avg_mort_30_cabg"),
        F.avg(F.when(F.col("measure_id") == "READM_30_AMI", F.col("score"))).alias("avg_readm_30_ami"),
        F.avg(F.when(F.col("measure_id") == "READM_30_HF", F.col("score"))).alias("avg_readm_30_hf"),
        F.avg(F.when(F.col("measure_id") == "READM_30_PN", F.col("score"))).alias("avg_readm_30_pn"),
        F.avg(F.when(F.col("measure_id") == "Hybrid_HWR", F.col("score"))).alias("avg_hospital_wide_readm"),
        F.avg(F.when(F.col("measure_id") == "H_HSP_RATING_9_10", F.col("score"))).alias("avg_pct_rating_9_10"),
        F.avg(F.when(F.col("measure_id") == "H_RECMND_DY", F.col("score"))).alias("avg_pct_definitely_recommend"),
    )
)

# Join all three pieces
gold_state_spending_quality = (
    spending_by_state
    .join(hospital_summary_by_state, on="state", how="left")
    .join(quality_by_state, on="state", how="left")
    .filter(F.col("state").isNotNull())
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    gold_state_spending_quality.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("state_spending_quality")
)

count = spark.table("state_spending_quality").count()
print(f"Wrote {count:,} rows to {CATALOG}.{GOLD_SCHEMA}.state_spending_quality")

Wrote 62 rows to medicare_provider_quality.gold.state_spending_quality


In [0]:
spark.sql("""
    SELECT 
        state,
        active_provider_count,
        ROUND(total_state_payment / 1e9, 2) AS total_state_payment_b,
        ROUND(payment_per_beneficiary, 0) AS payment_per_bene,
        hospital_count,
        ROUND(avg_hospital_overall_rating, 2) AS avg_star,
        ROUND(avg_mort_30_ami, 1) AS mort_ami,
        ROUND(avg_hospital_wide_readm, 1) AS readm_hwr,
        ROUND(avg_pct_rating_9_10, 1) AS pct_9_10
    FROM state_spending_quality
    WHERE state IN ('CA', 'TX', 'FL', 'NY', 'VA', 'AZ', 'WV', 'MA')
    ORDER BY total_state_payment_b DESC
""").show(truncate=False)

+-----+---------------------+---------------------+----------------+--------------+--------+--------+---------+--------+
|state|active_provider_count|total_state_payment_b|payment_per_bene|hospital_count|avg_star|mort_ami|readm_hwr|pct_9_10|
+-----+---------------------+---------------------+----------------+--------------+--------+--------+---------+--------+
|CA   |88508                |8.39                 |152.0           |277           |3.06    |11.9    |15.0     |66.4    |
|FL   |72688                |7.23                 |136.0           |167           |3.04    |12.1    |15.5     |66.8    |
|NY   |77374                |5.5                  |130.0           |129           |2.7     |11.7    |15.2     |63.5    |
|TX   |73505                |5.29                 |128.0           |285           |3.36    |12.3    |15.0     |72.7    |
|AZ   |22934                |2.49                 |170.0           |65            |3.02    |12.4    |14.9     |67.2    |
|VA   |28383                |2.1

## 4. gold.hospital_value_scorecard — one row per hospital, spending intensity + quality

The headline analytical table for the project's central business question: *does Medicare get what it pays for at the hospital level?* One row per hospital (CCN) joining hospital quality measures to the spending intensity of physicians who match to that hospital via the bridge.

### This is where the bridge actually matters

`gold.state_spending_quality` aggregated spending and quality independently because state grain doesn't need a row-level join. Hospital grain does. To say "Hospital X has high spending and low quality" or vice versa, I need to know which providers' spending counts toward Hospital X — and that's exactly what `silver.npi_to_ccn_bridge` gives me.

### The expected coverage

The bridge matched 1,951 of 3,115 acute care hospitals (62.6%). The scorecard will have ~1,951 rows. The ~1,164 unmatched hospitals will have quality data in silver but no spending-side data through the bridge, so they're excluded from this table. I'll flag this scope in the final report's limitations.

### What I'm computing per hospital

**Identity:**
- CCN, hospital name, state, ownership, overall star rating

**Spending intensity (the cost side):**
- Number of matched providers at this hospital
- Total Medicare payment across all matched providers
- Median provider payment (per-provider scale; uses median for the same reason as `specialty_benchmarks` — outliers exist)
- Average provider payment per beneficiary (the most meaningful "intensity" metric)

**Quality (the outcome side):**
- All 12 measures from `silver.hospital_quality`, pivoted to columns

**Derived value indicator:**
- A simple `value_quadrant` label: "high_cost_low_quality", "high_cost_high_quality", "low_cost_high_quality", "low_cost_low_quality" — based on whether the hospital is above/below the median on spending intensity and above/below the median composite mortality. This is the analytical hook for the dashboard scatter plot.

### The composite mortality score

The 6 individual mortality measures (AMI, HF, PN, COPD, stroke, CABG) each have different patient populations and slightly different scales. For the value quadrant, I'm computing an unweighted average of the 6 (treating missing measures as null and averaging across what's present). Hospitals with fewer than 3 of the 6 mortality measures populated get a null composite — they don't make it into the quadrant assignment.

Unweighted average is the right choice here. Weighting by patient volume would inflate the influence of AMI mortality (high-volume measure) over CABG mortality (low-volume but more variable across hospitals). Treating each measure equally is what CMS itself does in its star rating methodology.

### What this table doesn't try to do

No statistical correlation analysis or significance testing. That belongs in the analysis notebooks in Phase 5. This table is the input to those analyses, not the output.

In [0]:
# Build gold.hospital_value_scorecard — the headline analytical table

# --- Hospital identity and quality (from silver) ---
hospitals = spark.table(f"{SILVER_SCHEMA}.hospitals")

# Pivot quality measures to columns
quality_pivoted = (
    spark.table(f"{SILVER_SCHEMA}.hospital_quality")
    .groupBy("ccn")
    .agg(
        F.avg(F.when(F.col("measure_id") == "MORT_30_AMI", F.col("score"))).alias("mort_30_ami"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_HF", F.col("score"))).alias("mort_30_hf"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_PN", F.col("score"))).alias("mort_30_pn"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_COPD", F.col("score"))).alias("mort_30_copd"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_STK", F.col("score"))).alias("mort_30_stk"),
        F.avg(F.when(F.col("measure_id") == "MORT_30_CABG", F.col("score"))).alias("mort_30_cabg"),
        F.avg(F.when(F.col("measure_id") == "READM_30_AMI", F.col("score"))).alias("readm_30_ami"),
        F.avg(F.when(F.col("measure_id") == "READM_30_HF", F.col("score"))).alias("readm_30_hf"),
        F.avg(F.when(F.col("measure_id") == "READM_30_PN", F.col("score"))).alias("readm_30_pn"),
        F.avg(F.when(F.col("measure_id") == "Hybrid_HWR", F.col("score"))).alias("hospital_wide_readm"),
        F.avg(F.when(F.col("measure_id") == "H_HSP_RATING_9_10", F.col("score"))).alias("pct_rating_9_10"),
        F.avg(F.when(F.col("measure_id") == "H_RECMND_DY", F.col("score"))).alias("pct_definitely_recommend"),
    )
)

# Composite mortality: unweighted average of 6 measures
# Hospitals need at least 3 of 6 populated to get a composite
quality_with_composite = (
    quality_pivoted
    .withColumn(
        "mortality_measures_populated",
        F.coalesce(F.col("mort_30_ami").isNotNull().cast("int"), F.lit(0)) +
        F.coalesce(F.col("mort_30_hf").isNotNull().cast("int"),  F.lit(0)) +
        F.coalesce(F.col("mort_30_pn").isNotNull().cast("int"),  F.lit(0)) +
        F.coalesce(F.col("mort_30_copd").isNotNull().cast("int"), F.lit(0)) +
        F.coalesce(F.col("mort_30_stk").isNotNull().cast("int"), F.lit(0)) +
        F.coalesce(F.col("mort_30_cabg").isNotNull().cast("int"), F.lit(0))
    )
    .withColumn(
        "composite_mortality",
        F.when(
            F.col("mortality_measures_populated") >= 3,
            (F.coalesce(F.col("mort_30_ami"),  F.lit(0)) +
             F.coalesce(F.col("mort_30_hf"),   F.lit(0)) +
             F.coalesce(F.col("mort_30_pn"),   F.lit(0)) +
             F.coalesce(F.col("mort_30_copd"), F.lit(0)) +
             F.coalesce(F.col("mort_30_stk"),  F.lit(0)) +
             F.coalesce(F.col("mort_30_cabg"), F.lit(0))) / F.col("mortality_measures_populated")
        ).otherwise(None)
    )
    .drop("mortality_measures_populated")
)

# --- Spending intensity from bridge-matched providers ---
bridge = spark.table(f"{SILVER_SCHEMA}.npi_to_ccn_bridge")
provider_summary = spark.table("provider_spending_summary")

# Join bridge to provider_summary to get spending for each (CCN, provider) pair
matched_providers = (
    bridge.alias("b")
    .join(provider_summary.alias("p"), on="npi", how="inner")
    .select(
        F.col("b.ccn"),
        F.col("p.total_medicare_payment"),
        F.col("p.total_beneficiaries_unsuppressed"),
    )
)

# Aggregate to CCN level
spending_by_hospital = (
    matched_providers
    .groupBy("ccn")
    .agg(
        F.count("*").alias("matched_provider_count"),
        F.sum("total_medicare_payment").alias("total_matched_provider_payment"),
        F.expr("percentile_approx(total_medicare_payment, 0.50)").alias("median_provider_payment"),
        F.sum("total_beneficiaries_unsuppressed").alias("total_beneficiaries_unsuppressed"),
    )
    .withColumn(
        "avg_payment_per_beneficiary",
        F.when(F.col("total_beneficiaries_unsuppressed") > 0,
               F.col("total_matched_provider_payment") / F.col("total_beneficiaries_unsuppressed"))
         .otherwise(None)
    )
)

# --- Assemble the scorecard ---
scorecard = (
    hospitals.alias("h")
    .join(spending_by_hospital.alias("s"), on=F.col("h.ccn") == F.col("s.ccn"), how="inner")
    .join(quality_with_composite.alias("q"), on=F.col("h.ccn") == F.col("q.ccn"), how="left")
    .select(
        F.col("h.ccn"),
        F.col("h.hospital_name"),
        F.col("h.state"),
        F.col("h.city_original").alias("city"),
        F.col("h.hospital_ownership"),
        F.col("h.hospital_overall_rating"),
        # Spending
        F.col("s.matched_provider_count"),
        F.col("s.total_matched_provider_payment"),
        F.col("s.median_provider_payment"),
        F.col("s.avg_payment_per_beneficiary"),
        # Quality measures
        F.col("q.mort_30_ami"),
        F.col("q.mort_30_hf"),
        F.col("q.mort_30_pn"),
        F.col("q.mort_30_copd"),
        F.col("q.mort_30_stk"),
        F.col("q.mort_30_cabg"),
        F.col("q.readm_30_ami"),
        F.col("q.readm_30_hf"),
        F.col("q.readm_30_pn"),
        F.col("q.hospital_wide_readm"),
        F.col("q.pct_rating_9_10"),
        F.col("q.pct_definitely_recommend"),
        F.col("q.composite_mortality"),
    )
)

# Compute the value quadrant: above/below national median on spending and mortality
medians = scorecard.select(
    F.expr("percentile_approx(avg_payment_per_beneficiary, 0.50)").alias("median_spending"),
    F.expr("percentile_approx(composite_mortality, 0.50)").alias("median_mortality"),
).collect()[0]

median_spending = medians["median_spending"]
median_mortality = medians["median_mortality"]

scorecard_final = (
    scorecard
    .withColumn(
        "value_quadrant",
        F.when(
            F.col("avg_payment_per_beneficiary").isNull() | F.col("composite_mortality").isNull(),
            F.lit(None)
        )
        .when(
            (F.col("avg_payment_per_beneficiary") >= median_spending) &
            (F.col("composite_mortality") < median_mortality),
            F.lit("high_cost_high_quality")
        )
        .when(
            (F.col("avg_payment_per_beneficiary") >= median_spending) &
            (F.col("composite_mortality") >= median_mortality),
            F.lit("high_cost_low_quality")
        )
        .when(
            (F.col("avg_payment_per_beneficiary") < median_spending) &
            (F.col("composite_mortality") < median_mortality),
            F.lit("low_cost_high_quality")
        )
        .otherwise(F.lit("low_cost_low_quality"))
    )
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    scorecard_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hospital_value_scorecard")
)

count = spark.table("hospital_value_scorecard").count()
print(f"Wrote {count:,} rows to {CATALOG}.{GOLD_SCHEMA}.hospital_value_scorecard")
print(f"Median benchmarks used for quadrant assignment:")
print(f"  Spending intensity (payment per bene): ${median_spending:.0f}")
print(f"  Composite mortality rate:              {median_mortality:.2f}%")

Wrote 1,949 rows to medicare_provider_quality.gold.hospital_value_scorecard
Median benchmarks used for quadrant assignment:
  Spending intensity (payment per bene): $97
  Composite mortality rate:              11.68%


In [0]:
# Verify gold.hospital_value_scorecard and answer the headline question:
# How many hospitals fall into each value quadrant?

print("=== Quadrant distribution ===")
spark.sql("""
    SELECT 
        value_quadrant,
        COUNT(*) AS hospital_count,
        ROUND(AVG(avg_payment_per_beneficiary), 0) AS avg_spending_per_bene,
        ROUND(AVG(composite_mortality), 2) AS avg_composite_mortality,
        ROUND(AVG(hospital_overall_rating), 2) AS avg_star_rating
    FROM hospital_value_scorecard
    WHERE value_quadrant IS NOT NULL
    GROUP BY value_quadrant
    ORDER BY value_quadrant
""").show(truncate=False)

print("\n=== Top 5 'low_cost_high_quality' hospitals (the good news) ===")
spark.sql("""
    SELECT 
        hospital_name,
        state,
        city,
        hospital_overall_rating,
        ROUND(avg_payment_per_beneficiary, 0) AS pay_per_bene,
        ROUND(composite_mortality, 2) AS composite_mort
    FROM hospital_value_scorecard
    WHERE value_quadrant = 'low_cost_high_quality'
      AND hospital_overall_rating = 5
    ORDER BY composite_mortality ASC
    LIMIT 5
""").show(truncate=False)

print("\n=== Top 5 'high_cost_low_quality' hospitals (the value-failure story) ===")
spark.sql("""
    SELECT 
        hospital_name,
        state,
        city,
        hospital_overall_rating,
        ROUND(avg_payment_per_beneficiary, 0) AS pay_per_bene,
        ROUND(composite_mortality, 2) AS composite_mort
    FROM hospital_value_scorecard
    WHERE value_quadrant = 'high_cost_low_quality'
    ORDER BY composite_mortality DESC
    LIMIT 5
""").show(truncate=False)

=== Quadrant distribution ===
+----------------------+--------------+---------------------+-----------------------+---------------+
|value_quadrant        |hospital_count|avg_spending_per_bene|avg_composite_mortality|avg_star_rating|
+----------------------+--------------+---------------------+-----------------------+---------------+
|high_cost_high_quality|407           |129.0                |10.46                  |3.45           |
|high_cost_low_quality |386           |174.0                |12.95                  |2.78           |
|low_cost_high_quality |403           |74.0                 |10.41                  |3.53           |
|low_cost_low_quality  |430           |72.0                 |13.0                   |2.87           |
+----------------------+--------------+---------------------+-----------------------+---------------+


=== Top 5 'low_cost_high_quality' hospitals (the good news) ===
+------------------------------------------------------+-----+----------+---------------